# QuetHoaDon · Giai đoạn 0 của kế hoạch nâng độ chính xác

Mục tiêu: có **mốc chuẩn** để so sánh mọi cải tiến sau này. Chạy trên Kaggle (GPU T4, Internet bật), gắn dataset `quethoadon` và `m1b-v2-result`.
Chỉ dùng tập **validation**; không đọc test. Mỗi cell bắt đầu bằng `# CELL n`; dừng ở cell lỗi và dán output.

In [ ]:
# CELL 1 - Cài detectron2 (lần đầu 5-10 phút). Mong đợi: "detectron2 OK"
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

In [ ]:
# CELL 2 - Lấy mã nguồn. BRANCH = "main" sau khi đã merge nhánh HoaiNam/accuracy-plan; trước đó đặt đúng tên nhánh đó.
BRANCH = "main"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH} && git log --oneline -1
!pip install -q -r requirements-dev.txt

In [ ]:
# CELL 3 [KIỂM THỬ] - Tìm dữ liệu, kiểm tra đúng checkpoint đã chấm ở Tuần 4, chạy unit test.
# Mong đợi: số ảnh 1153; CKPT nằm trong runs/m1b_v2_20261110_183118; "Ran ... tests" rồi "OK"
import glob, os
TOK = os.path.dirname(glob.glob('/kaggle/input/**/token_dataset_v1*/**/validation.jsonl', recursive=True)[0])
IMG = os.path.dirname(glob.glob('/kaggle/input/**/preprocessed_images*/**/*.jpg', recursive=True)[0])
found = sorted(glob.glob('/kaggle/input/**/runs/m1b_v2_*/best', recursive=True))
CKPT = found[-1] if found else None
REF = glob.glob('/kaggle/input/**/eval/m1b_v2_val.json', recursive=True)[0]
print("TOK =", TOK, "->", sorted(os.listdir(TOK)))
print("IMG =", IMG, "| số ảnh:", len(os.listdir(IMG)))
print("CKPT =", CKPT, "| REF =", REF)
assert CKPT and os.path.basename(os.path.dirname(CKPT)) == "m1b_v2_20261110_183118", "checkpoint không phải run đã chấm ở Tuần 4"
os.makedirs('/kaggle/working/eval', exist_ok=True)
!python -m unittest discover -s tests 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"

In [ ]:
# CELL 4 [KIỂM THỬ] - Chấm lại M1b trên VALIDATION và lưu xác suất từng từ (khoảng 3 phút).
# Mong đợi: bảng điểm chịu lỗi OCR giống Tuần 4 (macro F1 validation 0.659) và dòng "word probabilities: ..."
!python scripts/evaluate_layoutxlm.py --checkpoint "{CKPT}" --token-dir "{TOK}" --image-dir "{IMG}" --split validation --save-probs /kaggle/working/eval/m1b_v2_val_probs.npz --out /kaggle/working/eval/m1b_v2_val_now.json 2>&1 | tail -22

In [ ]:
# CELL 5 [KIỂM THỬ] - Giải mã từ xác suất đã lưu (không GPU) và so với báo cáo vừa chấm ở cell 4.
# Mong đợi: dòng "reproduction: PASS" và bảng "Strict reading accuracy" (đây là MỐC CHUẨN của kế hoạch)
!python scripts/decode_from_probs.py --token-dir "{TOK}" --split validation --probs /kaggle/working/eval/m1b_v2_val_probs.npz --reference /kaggle/working/eval/m1b_v2_val_now.json --out /kaggle/working/eval/decoded_val.json

In [ ]:
# CELL 6 [KIỂM THỬ] - So báo cáo vừa chấm với báo cáo Tuần 4 trong dataset (cùng checkpoint, cùng validation).
# Mong đợi: hai macro F1 giống nhau; nếu REF là bản báo cáo cũ (thiếu một số khóa) cell chỉ nói rõ điều đó, không lỗi
import json, sys
sys.path.insert(0, '/kaggle/working/QuetHoaDon')
from src.evaluation.reproduction import reference_predictions, compare_predictions
a = json.load(open(REF)); b = json.load(open('/kaggle/working/eval/m1b_v2_val_now.json'))
FIELDS4 = ("SELLER", "ADDRESS", "TIMESTAMP", "TOTAL_COST")
def macro(r):
    f = r.get("fields_lenient_soft_rules")
    return None if f is None else round(sum(f[k]["f1"] for k in FIELDS4) / 4, 4)
print("khóa của REF:", sorted(a.keys())[:12], "...")
print("reference macro:", macro(a), "| now macro:", macro(b))
if "per_receipt_soft_rules" in a and "per_receipt_soft_rules" in b:
    print("khác biệt từng trường:", compare_predictions(reference_predictions(b), reference_predictions(a))["field_mismatches"])
else:
    print("REF là bản báo cáo cũ, không có dự đoán từng hóa đơn: không so được từng trường (cell 5 đã so với báo cáo vừa chấm)")

In [ ]:
# CELL 7 - Mốc chuẩn "đọc chuẩn" của M1b trên validation, đọc báo cáo VỪA CHẤM ở cell 4 (đủ khóa, không GPU).
# B0 và M1a trong dataset là báo cáo cũ; nếu cần cho chúng, chạy lại run_baseline_b0.py và đánh giá M1a rồi thêm --report.
!python scripts/score_app_metrics.py --token-dir "{TOK}" --split validation --report M1b=/kaggle/working/eval/m1b_v2_val_now.json --out /kaggle/working/eval/app_metrics_val.json --md /kaggle/working/eval/app_metrics_val.md

In [ ]:
# CELL 7b [KIỂM THỬ] - Chẩn đoán ngày và giờ (CPU, vài giây): ngày thiếu là do OCR hay do chọn, và giờ bị bỏ hay đọc sai?
# Mong đợi: một dict các số đếm; dán cho mình.
import json, sys, os
sys.path.insert(0, '/kaggle/working/QuetHoaDon')
from src.training.data import load_jsonl
from src.training.field_eval import gold_fields, normalize_date
from src.evaluation.app_metrics import extract_time
recs = load_jsonl(os.path.join(TOK, 'validation.jsonl')); gts = gold_fields(recs)
dec = json.load(open('/kaggle/working/eval/decoded_val.json'))
preds = {r["img_id"]: r["pred"] for r in dec["per_receipt_soft_rules"]}
s = dict(gold_with_date=0, date_correct=0, date_wrong=0, date_missing=0, date_missing_but_in_ocr=0,
         gold_with_time=0, pred_has_a_time=0, time_correct=0)
for rec in recs:
    g = " ".join(gts[rec["img_id"]].get("TIMESTAMP") or [])
    gd, gtime = normalize_date(g), extract_time(g)
    if gd is None:
        continue
    s["gold_with_date"] += 1
    p = preds[rec["img_id"]].get("TIMESTAMP") or ""
    pd_ = normalize_date(p)
    if pd_ is None:
        s["date_missing"] += 1
        s["date_missing_but_in_ocr"] += normalize_date(" ".join(rec["words"])) is not None
    elif pd_ == gd:
        s["date_correct"] += 1
    else:
        s["date_wrong"] += 1
    if gtime is not None:
        s["gold_with_time"] += 1
        s["pred_has_a_time"] += extract_time(p) is not None
        s["time_correct"] += extract_time(p) == gtime
print(s)

In [ ]:
# CELL 8 - Danh mục lỗi OCR cho việc xem tay (tự phân loại bước đầu). Mong đợi: bảng đếm theo trường và loại lỗi; tệp CSV để xem tay.
!python scripts/ocr_error_catalog.py --report /kaggle/working/eval/decoded_val.json --out-dir /kaggle/working/eval/ocr_errors

In [ ]:
# CELL 9 - Vẽ 30 hóa đơn validation sai để xem tay (mỗi ảnh có nguyên nhân tự động, chữ chuẩn và dự đoán). Mong đợi: "drew 30 receipts"
!python scripts/draw_errors.py --report /kaggle/working/eval/decoded_val.json --token-dir "{TOK}" --image-dir "{IMG}" --split validation --out /kaggle/working/error_images_val --limit 30

In [ ]:
# CELL 10 - Nén kết quả để tải về (tab Output → phase0_eval.zip) rồi mở các ảnh trong error_images_val và tệp CSV để xem tay.
!cd /kaggle/working && zip -q -r phase0_eval.zip eval error_images_val && ls -la phase0_eval.zip